# ORASR Quickstart

This starter notebook replaces the script-based demo with an interactive walkthrough of ORASR routing and gate behavior.

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent / 'src'))

In [ ]:
from orasr import GateType, ORASRRouter

## Router setup

In [ ]:
router = ORASRRouter(enable_fast_path=True, enable_audit=True)
router

## Pathway demo

In [ ]:
def simple_action(data):
    return {'status': 'success', 'value': data.get('value', 0) * 2}

low_result = router.route(action=simple_action, input_data={'value': 21}, risk_score=0.15)
mid_result = router.route(action=simple_action, input_data={'value': 21}, risk_score=0.55)
high_result = router.route(
    action=simple_action,
    input_data={'value': 21},
    risk_score=0.88,
    require_human_approval=True,
    human_approved=True,
)

[(r.path.name, r.safe, r.gates_passed, r.violations) for r in [low_result, mid_result, high_result]]

## Individual gate checks

In [ ]:
valid_context = {
    'input_data': {'patient_id': 'PT-001', 'value': 42},
    'risk_score': 0.5,
    'max_risk': 1.0,
    'action_result': {'status': 'success'},
}

gate_summary = {}
for gate_type in [
    GateType.PRECONDITION,
    GateType.RISK_ASSESSMENT,
    GateType.CONSTRAINT_VALIDATION,
    GateType.POSTCONDITION,
]:
    gate = router.gates[gate_type]
    result = gate.check(valid_context)
    gate_summary[gate.name] = {'passed': result.passed, 'message': result.message}

gate_summary

## Router statistics

In [ ]:
router.get_statistics()